# Cruce de mapas de riesgos con controles de ISOLUCION

### Importar librerías necesarias

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import process, fuzz


### Limpiar datos

#### Mapa de riesgos consolidado (desde Excel)

##### Cargar data de mapa de riesgos consolidado

In [2]:
# Leer archivo
mapa_excel = pd.read_excel('Consolidado_2025_v13.xlsx', sheet_name = 'Mapa Riesgos')

/home/jairoescrito/miniconda3/envs/jairoescrito/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


##### Limpiar data de mapa de riesgos consolidado

In [3]:
# Seleccionar variables de inetrés
mapa_ini = mapa_excel.iloc[9:,[0,2,3,6,9,10,11,13,14,15,29,30,31,32,33]]
# Eliminar filas en blanco
mapa_ini.dropna(how='all', inplace=True)
# Resetear índices 
mapa_ini = mapa_ini.reset_index(drop=True)
# Eliminar filas definitivas
mapa_ini = mapa_ini.iloc[:-3, :]

In [4]:
# Extraer nombres de columnas
columnas_mapa_ini = mapa_ini.iloc[0,:]
columnas_mapa_ini = columnas_mapa_ini.reset_index().iloc[:,1]
# Ajustar texto
columnas_mapa_ini = (columnas_mapa_ini.astype(str)
                    .str.lower()
                    .str.replace(" ", "_")
                    .str.replace("\n", ""))

In [5]:
# Cambiar algunos textos
columnas_mapa_ini.iloc[5] = "%_probabilidad_inherente"
columnas_mapa_ini.iloc[8] = "%_impacto_inherente"
columnas_mapa_ini.iloc[11] = "%_probabilidad_residual"
columnas_mapa_ini.iloc[13] = "%_impacto_residual"

In [6]:
# Cambiar los nombres de las columnas
mapa_ini.columns = columnas_mapa_ini
mapa_ini = mapa_ini.iloc[1:,:]

In [7]:
# Eliminar caracteres de espacios en blanco que están de mas
# Seleccionar las columnas que son tipo object (texto)
cols_texto = mapa_ini.select_dtypes(include='object').columns
# Para cada columna reemplazar el \n por espacio y luego eliminar espacios al inicio y al final 
mapa_ini[cols_texto] = (mapa_ini[cols_texto]
    .apply(lambda col: col.where(
        col.isna(),
        col.astype(str).str.replace(r'\\n|\n', ' ', regex=True).str.strip()
    ))
)

In [8]:
# Llenar los NaN hacia abajo
mapa_ini = mapa_ini.ffill()
# Cada riesgo se repite varias veces (segun la cantidad de controles)
# La última fila de cada riesgo tiene el valor final de riesgo residual
# Se selecciona de cada grupo 
mapa_ini = mapa_ini.groupby('consecutivo', sort=False).last().reset_index()

In [9]:
# Convertir a números las columnas que si son numéricas
for col in mapa_ini.select_dtypes(include='object').columns:
    mapa_ini[col] = pd.to_numeric(mapa_ini[col], errors='coerce').fillna(mapa_ini[col])

#### Mapa de riesgos desde controles ISOLUCION

##### Cargar data de reporte de controles ISOLUCION

In [10]:
# Leer archivo
mapa_iso =  pd.read_excel('Reporte Mejoramiento Continuo - Controles.xlsx', 
                        sheet_name = 'Data')

##### Limpiar data de reporte de riesgos

In [11]:
# Seleccionar solo las columnas relativas a los riesgos
mapa_rep = mapa_iso.iloc[:, [1, 2, 5, 7, 9]].copy()

In [12]:
# Eliminar las filas en blanco
mapa_rep.dropna(how='all', inplace=True)
# Resetear índices 
mapa_rep = mapa_rep.reset_index(drop=True)

In [13]:
# Eliminar mayúsculas y acentos en los nombres de las columnas
mapa_rep.columns = (mapa_rep.columns
                    .str.lower()
                    .str.normalize('NFKD')
                    .str.encode('ascii', errors='ignore')
                    .str.decode('utf-8'))

In [14]:
# Cambiar número de identificación del riesgo a entero
mapa_rep['num'] = mapa_rep['num'].astype(int)

## Cruce de riesgos identificados

### Limpieza adicional a las datas

In [15]:
# Función para limpiar texto y poder hacer la comparación del texto del riesgo en las dos tablas
def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    # Minúsculas
    texto = texto.lower()
    # Quitar tildes
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto)
                    if unicodedata.category(c) != 'Mn')
    # Quitar caracteres especiales
    texto = re.sub(r'[^a-z0-9\s]', '', texto)
    # Quitar espacios dobles
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

In [16]:
mapa_rep['proceso'] = mapa_rep['proceso'].str.strip()
mapa_rep['proceso'] = mapa_rep['proceso'].str.replace(r'\s+', ' ', regex=True)
mapa_rep['proceso'] = mapa_rep['proceso'].replace(
    'Gestión de Relacionamiento con el Ciudadano',
    'Gestión del Relacionamiento con el Ciudadano'
)

In [17]:
# Limpiar el texto de cada "texto de riesgo" en cada tabla (la que viene de los mapas y la que viene de ISOLUCION)
mapa_ini['desc_limpia'] = mapa_ini['descripción_del_riesgo'].apply(limpiar_texto)
mapa_rep['desc_limpia'] = mapa_rep['descripcion'].apply(limpiar_texto)
# Limpiar el texto de cada "proceso" en cada tabla(lo mismo, del mapa y de ISOLUCION)
mapa_ini['proceso_limpio'] = mapa_ini['proceso'].apply(limpiar_texto)
mapa_rep['proceso_limpio'] = mapa_rep['proceso'].apply(limpiar_texto)

In [18]:
# Eliminar duplicados en mapa_rep conservando el registro con mayor num (5461)
mapa_rep = mapa_rep[mapa_rep['num'] != 5454].reset_index(drop=True)

In [ ]:
# Planes de acción de controles pendientes de cierre
print(mapa_rep[['num','proceso']][mapa_rep['estado'] == 'Abierta'])

     num                                            proceso
0   5542                                 Asistencia Técnica
1   5543  Direccionamiento Estratégico y Articulación Te...
2   5544  Direccionamiento Estratégico y Articulación Te...
3   5546  Direccionamiento Estratégico y Articulación Te...
4   5549  Direccionamiento Estratégico y Articulación Te...
5   5551  Direccionamiento Estratégico y Articulación Te...
6   5552  Direccionamiento Estratégico y Articulación Te...
7   5555  Direccionamiento Estratégico y Articulación Te...
8   5560  Direccionamiento Estratégico y Articulación Te...
16  5484                 Promoción del Desarrollo Educativo
21  5489                 Promoción del Desarrollo Educativo
22  5490                 Promoción del Desarrollo Educativo
97  5504           Cultura Organizacional y Mejora Continua


### Cruce de datas

In [19]:
# Cruce de textos de riesgo para evaluar coincidencia de textos
# Los textos que se cargaron a ISOLUCION en algunos casos son diferentes a los que están en los mapas
# Les adicionaron texto, les cambiaron la ortografía, etc
choices_disponibles = mapa_rep.copy()
resultados_final = []

for proceso in mapa_ini['proceso'].unique():

    # Limpiar el texto del procesos a comparar
    proceso_limpio = limpiar_texto(proceso)

    # Riesgos del proceso en mapa_ini
    ini_proceso = mapa_ini[mapa_ini['proceso_limpio'] == proceso_limpio]
    
    # Riesgos disponibles del mismo proceso en mapa_rep
    rep_proceso = choices_disponibles[choices_disponibles['proceso_limpio'] == proceso_limpio]
    
    if rep_proceso.empty:
        # No hay nada en mapa_rep para este proceso
        for _, row in ini_proceso.iterrows():
            resultados_final.append({
                'consecutivo_ini': row['consecutivo'],
                'proceso': row['proceso'],
                'desc_ini': row['descripción_del_riesgo'],
                'num_rep': None,
                'proceso_rep': None,
                'desc_rep': None,
                'score_match': 0,
                'estado_match': 'sin_isolucion'
            })
        continue
    
    # Fuzzy dentro del proceso
    choices_proc = rep_proceso['desc_limpia'].tolist()
    asignados_rep = set()
    
    candidatos = []
    for _, row in ini_proceso.iterrows():
        matches = process.extract(
            row['desc_limpia'],
            choices_proc,
            scorer=fuzz.token_sort_ratio,
            limit=len(choices_proc)
        )
        for match in matches:
            candidatos.append({
                'consecutivo_ini': row['consecutivo'],
                'proceso': row['proceso'],
                'desc_ini': row['descripción_del_riesgo'],
                'match_idx_proc': match[2],
                'num_rep': rep_proceso.iloc[match[2]]['num'],
                'proceso_rep': rep_proceso.iloc[match[2]]['proceso'],
                'desc_rep': rep_proceso.iloc[match[2]]['descripcion'],
                'score_match': match[1]
            })
    
    # Greedy por proceso ordenado por score
    df_cand = pd.DataFrame(candidatos).sort_values('score_match', ascending=False)
    asignados_num = set()
    asignados_ini = set()
    
    for _, cand in df_cand.iterrows():
        if cand['consecutivo_ini'] not in asignados_ini and cand['num_rep'] not in asignados_num:
            cand['estado_match'] = 'automatico' if cand['score_match'] >= 85 \
                                   else 'revision' if cand['score_match'] >= 70 \
                                   else 'sin_match'
            resultados_final.append(cand.to_dict())
            asignados_ini.add(cand['consecutivo_ini'])
            asignados_num.add(cand['num_rep'])
    
    # Los que no encontraron match en este proceso
    ini_sin_match = set(ini_proceso['consecutivo']) - asignados_ini
    for consec in ini_sin_match:
        row = ini_proceso[ini_proceso['consecutivo'] == consec].iloc[0]
        resultados_final.append({
            'consecutivo_ini': consec,
            'proceso': row['proceso'],
            'desc_ini': row['descripción_del_riesgo'],
            'num_rep': None,
            'proceso_rep': None,
            'desc_rep': None,
            'score_match': 0,
            'estado_match': 'sin_isolucion'
        })



df_match = pd.DataFrame(resultados_final).sort_values('consecutivo_ini')

# Estas lineas son de forzado de asignación manual, excluir para mapas fuera de 2025
# Esta asignación forzada es por cambios de los textos de riesgos, sin actualización de mapa de riesgos
# Cuando fueron cargados a ISOLUCION
df_match.loc[df_match['consecutivo_ini'] == 60, 'estado_match'] = 'automatico'
df_match.loc[df_match['consecutivo_ini'] == 60, 'num_rep'] = 5551
df_match.loc[df_match['consecutivo_ini'] == 95, 'estado_match'] = 'automatico'
df_match.loc[df_match['consecutivo_ini'] == 92, 'estado_match'] = 'automatico'
df_match.loc[df_match['consecutivo_ini'] == 92, 'num_rep'] = 5469
df_match.loc[df_match['consecutivo_ini'] == 92, 'desc_rep'] = mapa_rep[mapa_rep['num'] == 5469]['descripcion'].values[0]

# Imprimir resultados de cruce de controles
print(df_match['estado_match'].value_counts())
print(f"\nTotal filas: {len(df_match)}")

estado_match
automatico       98
sin_isolucion     1
Name: count, dtype: int64

Total filas: 99
